In [1]:
# Install Brightway2 and required numerical dependencies compatible with Python 3.12+
!pip install "brightway2>=2.4.7" "bw2io>=0.8.12" pandas matplotlib seaborn

import brightway2 as bw
import bw2io as bi
import logging

logging.basicConfig(level=logging.INFO)

# 1. Create and switch to a dedicated Brightway project
project_name = "Tequila_LCA_Mexico"
bw.projects.set_current(project_name)
print(f"Current active Brightway project: {bw.projects.current}")

# 2. Setup the default biosphere and LCIA methodologies
if "biosphere3" not in bw.databases:
    print("Initializing biosphere3 database...")
    bw.bw2setup()
else:
    print("biosphere3 already present.")

# 3. Check / Import EXIOBASE or background database
db_name = "EXIOBASE_3"
if db_name not in bw.databases:
    print("EXIOBASE_3 database not present in project.")
    print("To load EXIOBASE, import your CSV/SUT datasets using bw2io.SingleOutputEcospold2Importer or standard SUT importers.")
else:
    print("EXIOBASE database initialized.")


In [ ]:
import brightway2 as bw

bw.projects.set_current("Tequila_LCA_Mexico")

def safe_search(db, query, location=None):
    """Helper to safely search Brightway databases with fallback handling."""
    results = db.search(query)
    if location:
        filtered = [r for r in results if r.get('location') == location]
        if filtered:
            return filtered[0]
    if results:
        return results[0]
    raise ValueError(f"Could not find query '{query}' (location={location}) in {db.name}")

# Initialize foreground database
fg_db = bw.Database("Tequila_Foreground")
fg_db.register()

# Primary functional unit activity
tequila_bottle = fg_db.new_activity(
    code="reposado_700ml",
    name="100% Reposado Tequila Bottle (700ml, 6-month aged)",
    unit="unit",
    location="MX"
)
tequila_bottle.save()

has_exio = "EXIOBASE_3" in bw.databases
exio = bw.Database("EXIOBASE_3") if has_exio else None
biosphere = bw.Database("biosphere3")

exchanges = [
    # Output
    {"input": tequila_bottle.key, "amount": 1.0, "type": "production"},
]

if has_exio:
    elec_mx = safe_search(exio, "Production of electricity", "MX")
    fuel_oil = safe_search(exio, "Production of fuel oil")
    water_supply = safe_search(exio, "Collection, purification and distribution of water")
    glass_bottle = safe_search(exio, "Manufacture of glass")
    co2_fossil = safe_search(biosphere, "Carbon dioxide, fossil")

    exchanges.extend([
        # 1. Agave Reception
        {"input": safe_search(exio, "Cultivation of crops", "MX").key, "amount": 8.62, "type": "technosphere"},
        {"input": elec_mx.key, "amount": 3.29e-03, "type": "technosphere"},
        # 2. Cooking
        {"input": fuel_oil.key, "amount": 8.06e-01, "type": "technosphere"},
        {"input": water_supply.key, "amount": 7.16e-01, "type": "technosphere"},
        {"input": elec_mx.key, "amount": 1.19e-04, "type": "technosphere"},
        # 3. Grinding
        {"input": elec_mx.key, "amount": 1.01e-01, "type": "technosphere"},
        # 4. Fermentation
        {"input": elec_mx.key, "amount": 9.36e-02, "type": "technosphere"},
        {"input": safe_search(exio, "Manufacture of food products").key, "amount": 4.44e-03, "type": "technosphere"},
        {"input": co2_fossil.key, "amount": 3.17e-02, "type": "biosphere"},
        # 5. Distillation
        {"input": fuel_oil.key, "amount": 2.12e-03 + 1.33e-01, "type": "technosphere"},
        {"input": elec_mx.key, "amount": 1.96e-01 + 8.08e-01, "type": "technosphere"},
        # 6. Rectification & Aging
        {"input": water_supply.key, "amount": 1.97e-01, "type": "technosphere"},
        {"input": elec_mx.key, "amount": 5.24e-04 + 7.21e-04 + 3.50e-04, "type": "technosphere"},
        {"input": safe_search(exio, "Manufacture of chemicals").key, "amount": 6.35e-06 + 1.13e-05, "type": "technosphere"},
        # 7. Bottling & Packaging
        {"input": elec_mx.key, "amount": 5.07e-04, "type": "technosphere"},
        {"input": glass_bottle.key, "amount": 5.50e-01, "type": "technosphere"},
        {"input": safe_search(exio, "Manufacture of aluminum").key, "amount": 9.80e-02, "type": "technosphere"},
        {"input": safe_search(exio, "Manufacture of wood products").key, "amount": 2.45e-01, "type": "technosphere"},
    ])

for exchange in exchanges:
    tequila_bottle.new_exchange(**exchange).save()

print(f"Tequila activity '{tequila_bottle['name']}' built with {len(tequila_bottle.exchanges())} exchanges.")


In [ ]:
# Define target methods based on your study requirements
cml_gwp = [m for m in bw.methods if "CML" in m[0] and "global warming" in m[1]][0]
recipe_endpoint = [m for m in bw.methods if "ReCiPe" in m[0] and "Endpoint" in m[1]][0]

# Perform calculation
lca = bw.LCA({tequila_bottle: 1}, cml_gwp)
lca.lci()
lca.lcia()

print(f"Replicated Climate Footprint (CML): {lca.score:.4f} kg CO2-eq per bottle")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import brightway2 as bw

# Ensure the active workspace is correct
bw.projects.set_current("Tequila_LCA_Mexico")
tequila_activity = bw.Database("Tequila_Foreground").search("Reposado Tequila")[0]

# --- 1. NUMERICAL OUTPUT: MULTI-METHOD SUMMARY ---
selected_methods = [
    ("CML-IA baseline", "global warming", "GWP100a"),
    ("CML-IA baseline", "abiotic depletion", "fossil fuels"),
    ("CML-IA baseline", "human toxicity", "human toxicity"),
    ("ReCiPe Endpoint (H)", "Total", "Total Endpoint Impact")
]

summary_data = []
for m in selected_methods:
    try:
        target_method = [method for method in bw.methods if m[0] in method[0] and m[1] in method[1]][0]
        lca = bw.LCA({tequila_activity: 1}, target_method)
        lca.lci()
        lca.lcia()
        summary_data.append({
            "Method Family": m[0],
            "Impact Category": m[1],
            "Score": lca.score,
            "Unit": bw.methods[target_method].get('unit', 'unknown')
        })
    except IndexError:
        continue

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv("tequila_lcia_totals.csv", index=False)
print("Saved total impact summary to CSV.")


# --- 2. NUMERICAL & GRAPHICAL OUTPUT: HOTSPOT CONTRIBUTION ---
# Focus heavily on Carbon Footprint (GWP100a)
gwp_method = [m for m in bw.methods if "CML" in m[0] and "global warming" in m[1]][0]
lca_gwp = bw.LCA({tequila_activity: 1}, gwp_method)
lca_gwp.lci()
lca_gwp.lcia()

# Traverse immediate foreground exchanges
contribution_data = []
for exc in tequila_activity.exchanges():
    if exc['type'] == 'production':
        continue
    # Redo calculation isolating this single structural stream
    lca_gwp.redo_lcia({exc.input: exc['amount']})
    contribution_data.append({
        "Stage": exc.input['name'],
        "Absolute GWP (kg CO2-eq)": lca_gwp.score
    })

df_contrib = pd.DataFrame(contribution_data)
# Add percentage calculation
total_gwp_score = df_contrib["Absolute GWP (kg CO2-eq)"].sum()
df_contrib["Percentage Contribution (%)"] = (df_contrib["Absolute GWP (kg CO2-eq)"] / total_gwp_score) * 100
df_contrib = df_contrib.sort_values(by="Absolute GWP (kg CO2-eq)", ascending=False)

# Save Table
df_contrib.to_csv("tequila_gwp_process_contribution.csv", index=False)

# Render Chart
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_contrib.head(6),  # Plot top 6 hotspots
    y="Stage",
    x="Percentage Contribution (%)",
    palette="viridis"
)
plt.title("Carbon Footprint (GWP100a) Foreground Hotspot Analysis")
plt.xlabel("Share of Total Global Warming Impact (%)")
plt.ylabel("Lifecycle Process Stage")
plt.tight_layout()

# Save Visual
plt.savefig("tequila_hotspot_chart.png", dpi=300)
plt.close()
print("Saved process contribution tables and visualization charts successfully.")
